In [8]:
from pydantic import BaseModel
from typing import List



# BUILDING PYDANTIC SCHEMA FOR THE SEASHELL STORE

class PurchasedProduct(BaseModel):
    product_name: str
    quantity: int
    price: float


class CustomerOrder(BaseModel):
    order_id: int
    status: str
    total: float
    products: List[PurchasedProduct]


class CustomerDocument(BaseModel):
    customer_id: int
    name: str
    email: str
    city: str
    orders: List[CustomerOrder]

In [9]:
import sqlite3

conn = sqlite3.connect("../../shop.db")
cursor = conn.cursor()


# BUILDING LIST OF COMPLETE CUSTOMER DOCUMENTS FROM DB DATA

cursor.execute("""
    SELECT id, name, email, city
    FROM customers
""")

customers = cursor.fetchall()


documents = []


for customer_id, name, email, city in customers:

    cursor.execute("""
        SELECT
            o.id,
            o.status,
            o.total,
            oi.quantity,
            oi.price,
            p.name
        FROM orders o
        JOIN order_items oi
            ON o.id = oi.order_id
        JOIN products p
            ON oi.product_id = p.id
        WHERE o.customer_id = ?
        ORDER BY o.id
    """, (customer_id,))

    rows = cursor.fetchall()


    orders = {}

    for (
        order_id,
        status,
        total,
        quantity,
        price,
        product_name,
    ) in rows:

        if order_id not in orders:
            orders[order_id] = CustomerOrder(
                order_id=order_id,
                status=status,
                total=total,
                products=[],
            )

        orders[order_id].products.append(
            PurchasedProduct(
                product_name=product_name,
                quantity=quantity,
                price=price,
            )
        )


    customer_document = CustomerDocument(
        customer_id=customer_id,
        name=name,
        email=email,
        city=city,
        orders=list(orders.values()),
    )

    documents.append(customer_document)


conn.close()

In [10]:
# Convert them to JSON

import json

data = [
    document.model_dump()
    for document in documents
]

with open("customers.json", "w") as f:
    json.dump(data, f, indent=2)

### NOW that we have a json document for our data we can start data ingestion and creating documents

In [11]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="customers.json",
    jq_schema=".[]",
    text_content=False,
)

docs = loader.load()

/tmp/ipykernel_40167/1551820869.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader


In [12]:
for doc in docs[:2]:
    print(doc)
    print("=" * 80)

page_content='{"customer_id": 1, "name": "Sadaf Iqbal", "email": "sadafiqbal4911@gmail.com", "city": "Kabirwala", "orders": [{"order_id": 1, "status": "completed", "total": 661, "products": [{"product_name": "Mechanical Keyboard", "quantity": 3, "price": 120}, {"product_name": "Bluetooth Speaker", "quantity": 1, "price": 85}, {"product_name": "Ethernet Cable", "quantity": 3, "price": 12}, {"product_name": "Webcam", "quantity": 2, "price": 90}]}, {"order_id": 2, "status": "processing", "total": 615, "products": [{"product_name": "Laptop Stand", "quantity": 1, "price": 75}, {"product_name": "1080p Monitor", "quantity": 3, "price": 180}]}, {"order_id": 3, "status": "processing", "total": 40, "products": [{"product_name": "Smartphone Stand", "quantity": 2, "price": 20}]}, {"order_id": 4, "status": "shipped", "total": 120, "products": [{"product_name": "USB-C Hub", "quantity": 2, "price": 60}]}]}' metadata={'source': '/home/workspace/learning/RAG/notebooks/seashell_store_data_ingestion/cust

In [ ]:
# This is what current metadata looks like

metadata={'source': '/home/workspace/learning/RAG/notebooks/seashell_store_data_ingestion/customers.json', 'seq_num': 1}

### OK we have the documents but the metadata isn't that useful so we fine tune it

In [13]:
def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["customer_id"] = record["customer_id"]
    metadata["customer_name"] = record["name"]
    metadata["city"] = record["city"]
    metadata["email"] = record["email"]

    metadata["order_count"] = len(record["orders"])

    return metadata

In [14]:
loader = JSONLoader(
    file_path="customers.json",
    jq_schema=".[]",
    content_key=None,
    metadata_func=metadata_func,
    text_content=False,
)

In [15]:
docs = loader.load()

In [16]:
for doc in docs[:2]:
    print("PAGE CONTENT:")
    print(doc.page_content)

    print("\nMETADATA:")
    print(doc.metadata)

    print("=" * 80)

PAGE CONTENT:
{"customer_id": 1, "name": "Sadaf Iqbal", "email": "sadafiqbal4911@gmail.com", "city": "Kabirwala", "orders": [{"order_id": 1, "status": "completed", "total": 661, "products": [{"product_name": "Mechanical Keyboard", "quantity": 3, "price": 120}, {"product_name": "Bluetooth Speaker", "quantity": 1, "price": 85}, {"product_name": "Ethernet Cable", "quantity": 3, "price": 12}, {"product_name": "Webcam", "quantity": 2, "price": 90}]}, {"order_id": 2, "status": "processing", "total": 615, "products": [{"product_name": "Laptop Stand", "quantity": 1, "price": 75}, {"product_name": "1080p Monitor", "quantity": 3, "price": 180}]}, {"order_id": 3, "status": "processing", "total": 40, "products": [{"product_name": "Smartphone Stand", "quantity": 2, "price": 20}]}, {"order_id": 4, "status": "shipped", "total": 120, "products": [{"product_name": "USB-C Hub", "quantity": 2, "price": 60}]}]}

METADATA:
{'source': '/home/workspace/learning/RAG/notebooks/seashell_store_data_ingestion/cus

In [17]:
# This is how after fine tuning it looks like


METADATA= {'source': '/home/workspace/learning/RAG/notebooks/seashell_store_data_ingestion/customers.json', 'seq_num': 1, 'customer_id': 1, 'customer_name': 'Sadaf Iqbal', 'city': 'Kabirwala', 'email': 'sadafiqbal4911@gmail.com', 'order_count': 4}

In [18]:
print(len(docs))

100


### CHUNKING

In [19]:
# chunking on per order per customer
# we are not using this RecursiveCharacterTextSplitter
# because our data is structured not some random pdfs and we need to blindly split the text into chunks

import json

from langchain_core.documents import Document


def create_customer_chunks(documents):
    chunks = []

    for document in documents:
        customer = json.loads(document.page_content)

        customer_id = customer["customer_id"]
        customer_name = customer["name"]
        email = customer["email"]
        city = customer["city"]

        for order in customer["orders"]:
            products = "\n".join(
                f"- {product['product_name']}: "
                f"{product['quantity']} × ${product['price']}"
                for product in order["products"]
            )

            content = f"""
                Customer: {customer_name}
                Customer ID: {customer_id}
                Email: {email}
                City: {city}

                Order ID: {order['order_id']}
                Status: {order['status']}
                Total: ${order['total']}

                Products:
                {products}
            """.strip()

            metadata = {
                "source": document.metadata["source"],
                "customer_id": customer_id,
                "customer_name": customer_name,
                "city": city,
                "email": email,
                "order_id": order["order_id"],
                "order_status": order["status"],
            }

            chunks.append(
                Document(
                    page_content=content,
                    metadata=metadata
                )
            )

    return chunks

In [20]:
chunks = create_customer_chunks(docs)

for chunk in chunks:
    print("=" * 80)
    print(chunk.page_content)
    print(chunk.metadata)

Customer: Sadaf Iqbal
                Customer ID: 1
                Email: sadafiqbal4911@gmail.com
                City: Kabirwala

                Order ID: 1
                Status: completed
                Total: $661

                Products:
                - Mechanical Keyboard: 3 × $120
- Bluetooth Speaker: 1 × $85
- Ethernet Cable: 3 × $12
- Webcam: 2 × $90
{'source': '/home/workspace/learning/RAG/notebooks/seashell_store_data_ingestion/customers.json', 'customer_id': 1, 'customer_name': 'Sadaf Iqbal', 'city': 'Kabirwala', 'email': 'sadafiqbal4911@gmail.com', 'order_id': 1, 'order_status': 'completed'}
Customer: Sadaf Iqbal
                Customer ID: 1
                Email: sadafiqbal4911@gmail.com
                City: Kabirwala

                Order ID: 2
                Status: processing
                Total: $615

                Products:
                - Laptop Stand: 1 × $75
- 1080p Monitor: 3 × $180
{'source': '/home/workspace/learning/RAG/notebooks/seashell_

In [21]:
len(chunks)

309

### EMBEDDING

In [22]:
import chromadb
from sentence_transformers import SentenceTransformer


# --------------------------------------------------
# 1. Load embedding model
# --------------------------------------------------

model = SentenceTransformer("all-MiniLM-L6-v2")


# --------------------------------------------------
# 2. ChromaDB
# --------------------------------------------------

client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = client.get_or_create_collection(
    name="customer_orders"
)


# --------------------------------------------------
# 3. Prepare chunks
# --------------------------------------------------

documents = [
    chunk.page_content
    for chunk in chunks
]

metadatas = [
    chunk.metadata
    for chunk in chunks
]

ids = [
    f"chunk_{index}"
    for index in range(len(chunks))
]


# --------------------------------------------------
# 4. Create embeddings
# --------------------------------------------------

embeddings = model.encode(documents).tolist()


# --------------------------------------------------
# 5. Store in ChromaDB
# --------------------------------------------------

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
)


print(f"Stored {len(documents)} chunks")

/home/workspace/learning/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8680.02it/s]


Stored 309 chunks


In [25]:
query = "What did Sadaf order?"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
    where={
        "customer_name": "Sadaf Iqbal"
    },
    include=["documents", "metadatas", "distances"],
)

for i, document in enumerate(results["documents"][0]):
    print("=" * 60)
    print("Distance:", results["distances"][0][i])
    print(document)
    print("Metadata:", results["metadatas"][0][i])

Distance: 1.6320019960403442
Customer: Sadaf Iqbal
                Customer ID: 1
                Email: sadafiqbal4911@gmail.com
                City: Kabirwala

                Order ID: 3
                Status: processing
                Total: $40

                Products:
                - Smartphone Stand: 2 × $20
Metadata: {'customer_id': 1, 'customer_name': 'Sadaf Iqbal', 'city': 'Kabirwala', 'order_status': 'processing', 'order_id': 3, 'source': '/home/workspace/learning/RAG/notebooks/seashell_store_data_ingestion/customers.json', 'email': 'sadafiqbal4911@gmail.com'}
Distance: 1.693780541419983
Customer: Sadaf Iqbal
                Customer ID: 1
                Email: sadafiqbal4911@gmail.com
                City: Kabirwala

                Order ID: 1
                Status: completed
                Total: $661

                Products:
                - Mechanical Keyboard: 3 × $120
- Bluetooth Speaker: 1 × $85
- Ethernet Cable: 3 × $12
- Webcam: 2 × $90
Metadata: {'cus

In [26]:
import chromadb

from sentence_transformers import SentenceTransformer
from langchain_ollama import ChatOllama


# --------------------------------------------------
# 1. Load embedding model
# --------------------------------------------------

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# --------------------------------------------------
# 2. Connect to ChromaDB
# --------------------------------------------------

client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = client.get_collection(
    name="customer_orders"
)


# --------------------------------------------------
# 3. Load LLM
# --------------------------------------------------

llm = ChatOllama(
    model="qwen3.5:4b",
    temperature=0
)


# --------------------------------------------------
# 4. User question
# --------------------------------------------------

query = "What did Sadaf order?"


# --------------------------------------------------
# 5. Create query embedding
# --------------------------------------------------

query_embedding = embedding_model.encode(
    query
).tolist()


# --------------------------------------------------
# 6. Retrieve relevant chunks
# --------------------------------------------------

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
    where={
        "customer_name": "Sadaf Iqbal"
    },
    include=[
        "documents",
        "metadatas",
    ],
)


# --------------------------------------------------
# 7. Build context
# --------------------------------------------------

context = "\n\n".join(
    results["documents"][0]
)


# --------------------------------------------------
# 8. Create prompt
# --------------------------------------------------

prompt = f"""
You are a helpful shop assistant.

Answer the user's question using ONLY the information
provided in the context.

If the answer is not present in the context,
say that you don't have enough information.

Context:
{context}

User question:
{query}

Answer:
"""


# --------------------------------------------------
# 9. Ask the LLM
# --------------------------------------------------

response = llm.invoke(prompt)


print("\nAnswer:")
print(response.content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4936.00it/s]



Answer:
Based on the context provided, Sadaf Iqbal has placed four orders containing the following products:

**Order ID 1 (Status: completed)**
*   Mechanical Keyboard (3 × $120)
*   Bluetooth Speaker (1 × $85)
*   Ethernet Cable (3 × $12)
*   Webcam (2 × $90)

**Order ID 2 (Status: processing)**
*   Laptop Stand (1 × $75)
*   1080p Monitor (3 × $180)

**Order ID 3 (Status: processing)**
*   Smartphone Stand (2 × $20)

**Order ID 4 (Status: shipped)**
*   USB-C Hub (2 × $60)


In [30]:
from langchain_core.tools import tool


@tool
def search_customer_orders(
    query: str,
    customer_name: str
) -> str:
    """
    Search customer order information using semantic search.
    Use this when the user asks about products, orders, or
    other information contained in the customer knowledge base.
    """

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        where={
            "customer_name": customer_name
        },
    )

    documents = results["documents"][0]

    return "\n\n".join(documents)

In [31]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[search_customer_orders],
)

In [35]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What did Faaiz Ali Tariq order?"
        }
    ]
})

print(response)

{'messages': [HumanMessage(content='What did Faaiz Ali Tariq order?', additional_kwargs={}, response_metadata={}, id='9b349ee4-c0f8-4515-8bfc-3c1f277173eb'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-10-01T11:59:03.730403616Z', 'done': True, 'done_reason': 'stop', 'total_duration': 49266607249, 'load_duration': 9155311599, 'prompt_eval_count': 316, 'prompt_eval_duration': 17332535000, 'eval_count': 148, 'eval_duration': 22772758000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0f754-55be-7661-a43c-9faac3f83ce2-0', tool_calls=[{'name': 'search_customer_orders', 'args': {'query': 'Faaiz Ali Tariq order', 'customer_name': 'Faaiz Ali Tariq'}, 'id': '5b9d5796-dc90-4ca1-bc68-a32868f46046', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 316, 'output_tokens': 148, 'total_tokens': 464}), ToolMessage(content='Customer: Faaiz Ali Tariq\n                Custom